# DiffPlanner floor-plan generator

Run this notebook in Google Colab. In **Runtime → Change runtime type**, select a GPU for faster generation. The notebook clones the project, installs its app dependencies, downloads the trained checkpoint archive, extracts the three required weights into the expected locations, and opens the existing Gradio interface.

The three checkpoints are required to generate floor plans. The trained model is not included in this repository.

In [ ]:
# Confirm whether Colab assigned a GPU. A CPU runtime can run the app, but sampling will be slower.
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (CPU runtime)')

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/DiffPlanner')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/djamal-m/DiffPlanner.git', str(repo)], check=True)
print('DiffPlanner revision:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip())
%cd /content/DiffPlanner
%pip install -r requirements-app.txt

## Download the trained checkpoints

The notebook downloads the trained model archive published with DiffPlanner and places its three checkpoints in the paths expected by the inference app. No Drive mount or manual checkpoint upload is needed.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import os

repo = Path('/content/DiffPlanner')
archive = Path('/content/trained_model.zip')
model_url = 'https://github.com/shidong-wang/DiffPlanner/releases/download/trained_model/trained_model.zip'
print('Downloading trained model archive...')
urlretrieve(model_url, archive)

members = {
    'DIFFPLANNER_NODE_CHECKPOINT': ('trained_model/node_diff/scripts/trained_model/b_model300000.pt', 'node_diff/scripts/trained_model/b_model300000.pt'),
    'DIFFPLANNER_ADJACENCY_CHECKPOINT': ('trained_model/adjacency_diff/scripts/trained_model/bncsl_model300000.pt', 'adjacency_diff/scripts/trained_model/bncsl_model300000.pt'),
    'DIFFPLANNER_PARTITION_CHECKPOINT': ('trained_model/partitioning_diff/scripts/trained_model/bncsla_model300000.pt', 'partitioning_diff/scripts/trained_model/bncsla_model300000.pt'),
}
with ZipFile(archive) as zf:
    available = set(zf.namelist())
    for env_name, (archive_name, relative_path) in members.items():
        if archive_name not in available:
            raise FileNotFoundError(f'Checkpoint missing from release archive: {archive_name}')
        checkpoint = repo / relative_path
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        checkpoint.write_bytes(zf.read(archive_name))
        os.environ[env_name] = str(checkpoint)

for key in ('DIFFPLANNER_NODE_CHECKPOINT', 'DIFFPLANNER_ADJACENCY_CHECKPOINT', 'DIFFPLANNER_PARTITION_CHECKPOINT'):
    checkpoint = Path(os.environ[key])
    print(f'{key}: {checkpoint} ({checkpoint.stat().st_size:,} bytes)')

## Launch the interface

Run this cell and open the Gradio link it prints. Enter boundary vertices and the entrance as `[x, y]` pixel coordinates on the 256 × 256 canvas. Results show the room bubbles, predicted adjacencies, and aligned floor plan. The app supports 1–5 distinct diffusion samples per request.

In [ ]:
import importlib
import inference.pipeline
import api.app

# Colab keeps imported modules cached between cell runs; reload them after syncing the repository.
importlib.reload(inference.pipeline)
app_module = importlib.reload(api.app)
print('Launching app from:', app_module.__file__)
demo = app_module.demo
demo.launch(share=True, show_error=True)